# Patrones de memoria en sistemas multi-agente
1. Message passing (`call_agent`)
2. Memory reflection (`call_agent_with_reflection`)
3. Memory handoff (`hand_off_to_agent`)
4. Selective memory sharing (`call_agent_with_selected_context`)

In [10]:
%run "4.1_frameworkAgent.ipynb"

In [11]:
# registro simple de agentes
class AgentRegistry:
    def __init__(self):
        self.agents = {}

    def register_agent(self, name: str, run_function):
        self.agents[name] = run_function

    def get_agent(self, name: str):
        return self.agents.get(name)

In [12]:
# agente especialista
# agente sencilla que solo analiza el texto y responde, concluye directamente con un analisis
def create_agente_analista():
    action_registry = PythonActionRegistry(tags=["system"])  # solo trae terminate
    environment = PythonEnvironment()

    goals = [
        Goal(
            name="Persona",
            description="Eres un analista breve y directo."
        ),
        Goal(
            name="Analizar",
            description="Analiza la tarea que te den y responde con tu conclusión usando terminate."
        )
    ]

    return Agent(
        goals=goals,
        agent_language=AgentFunctionCallingActionLanguage(),
        action_registry=action_registry,
        generate_response=generate_response,
        environment=environment
    )

agente_analista = create_agente_analista()

registry = AgentRegistry()
registry.register_agent("agente_analista", agente_analista.run)

In [13]:
# los 4 tools de cordinacion adapatadas al framework
def _extract_text(memory_item: dict) -> str:
    """
    El framework a veces guarda el content del último item como un dict
    anidado (ej. {"tool_executed": true, "result": "texto real", ...})
    en vez de puro texto. Esta función lo desenvuelve para quedarnos
    solo con el mensaje final.
    """
    content = memory_item.get("content", "Sin resultado")
    while isinstance(content, dict) and "result" in content:
        content = content["result"]
    return content


def call_agent(action_context: ActionContext, agent_name: str, task: str) -> dict:
    """Patrón 1: Message passing — memoria nueva y limpia, solo regresa el resultado final."""
    agent_registry = action_context.get("agent_registry")
    agent_run = agent_registry.get_agent(agent_name)

    invoked_memory = Memory()
    result_memory = agent_run(user_input=task, memory=invoked_memory)

    return {"result": _extract_text(result_memory.items[-1])}


def call_agent_with_reflection(action_context: ActionContext, agent_name: str, task: str) -> dict:
    """Patrón 2: Memory reflection — copia todo el proceso del agente invocado a la memoria del que llama."""
    agent_registry = action_context.get("agent_registry")
    agent_run = agent_registry.get_agent(agent_name)

    invoked_memory = Memory()
    result_memory = agent_run(user_input=task, memory=invoked_memory)

    caller_memory = action_context.get("memory")
    for memory_item in result_memory.items:
        caller_memory.add_memory({
            "type": f"{agent_name}_thought",
            "content": memory_item["content"]
        })

    return {"result": _extract_text(result_memory.items[-1]),
            "memories_added": len(result_memory.items)}


def hand_off_to_agent(action_context: ActionContext, agent_name: str, task: str) -> dict:
    """Patrón 3: Memory handoff — el especialista recibe la MISMA memoria, con todo el historial previo."""
    agent_registry = action_context.get("agent_registry")
    agent_run = agent_registry.get_agent(agent_name)

    current_memory = action_context.get("memory")
    result_memory = agent_run(user_input=task, memory=current_memory)

    return {"result": _extract_text(result_memory.items[-1])}


def call_agent_with_selected_context(action_context: ActionContext, agent_name: str, task: str) -> dict:
    """Patrón 4: Selective memory sharing — el LLM elige qué memorias son relevantes para la tarea."""
    agent_registry = action_context.get("agent_registry")
    agent_run = agent_registry.get_agent(agent_name)

    current_memory = action_context.get("memory")
    memory_with_ids = [{**item, "memory_id": f"mem_{i}"} for i, item in enumerate(current_memory.items)]

    selection_schema = {
        "type": "object",
        "properties": {
            "selected_memories": {"type": "array", "items": {"type": "string"}},
            "reasoning": {"type": "string"}
        },
        "required": ["selected_memories", "reasoning"]
    }

    memory_text = "\n".join(f"Memoria {m['memory_id']}: {m['content']}" for m in memory_with_ids)

    selection_prompt = f"""Revisa estas memorias y selecciona las relevantes para esta tarea:

Tarea: {task}

Memorias disponibles:
{memory_text}

Selecciona las memorias que dan contexto importante para esta tarea específica."""

    selection = prompt_llm_for_json(
        action_context=action_context,
        schema=selection_schema,
        prompt=selection_prompt
    )

    filtered_memory = Memory()
    selected_ids = set(selection["selected_memories"])
    for item in memory_with_ids:
        if item["memory_id"] in selected_ids:
            item_copy = item.copy()
            del item_copy["memory_id"]
            filtered_memory.add_memory(item_copy)

    result_memory = agent_run(user_input=task, memory=filtered_memory)

    return {
        "result": _extract_text(result_memory.items[-1]),
        "shared_memories": len(filtered_memory.items),
        "selection_reasoning": selection["reasoning"]
    }

In [14]:
# prompt_llm_for_json es una función auxiliar para interactuar con el LLM y obtener una respuesta en formato JSON
def prompt_llm_for_json(action_context: ActionContext, schema: dict, prompt: str):
    llm = action_context.get("llm")
    for i in range(3):
        try:
            response = llm(Prompt(messages=[
                {"role": "system",
                 "content": f"Debes producir una salida que cumpla con el siguiente JSON schema:\n\n"
                            f"{json.dumps(schema, indent=4)}. Escribe tu JSON dentro de un bloque markdown ```json."},
                {"role": "user", "content": prompt}
            ]))
            if "```json" in response:
                start = response.find("```json")
                end = response.rfind("```")
                response = response[start + 7:end].strip()
            return json.loads(response)
        except Exception as e:
            if i == 2:
                raise e
            print(f"Reintentando... ({e})")
            

In [16]:
# patron #1 Message passing
# se crea la memoria del coordinador, se delega una tarea sin exponer el historial al especialista
coordinador_memory = Memory()
coordinador_memory.add_memory({"type": "user", "content": "Necesitamos armar el reporte trimestral"})

action_context = ActionContext({
    "llm": generate_response,
    "agent_registry": registry,
    "memory": coordinador_memory
})

resultado = call_agent(action_context, "agente_analista", "Analiza brevemente los riesgos de lanzar un producto sin pruebas de usuario")
print("Resultado:", resultado)
print("Memoria del coordinador (no cambió):", len(coordinador_memory.items), "items")

Agent thinking...
Agent Decision: {"tool": "terminate", "args": {"message": "Riesgos al lanzar sin pruebas de usuario: 1) Fallos de usabilidad que provocan abandono y mala reputaci\u00f3n; 2) Errores cr\u00edticos no detectados que generan costes de soporte y posibles da\u00f1os legales; 3) Desalineaci\u00f3n con necesidades reales del mercado, resultando en baja adopci\u00f3n y p\u00e9rdida de inversi\u00f3n; 4) Feedback limitado impide iteraciones r\u00e1pidas, dificultando mejoras posteriores. En resumen, lanzar sin pruebas expone al producto a problemas de calidad, aceptaci\u00f3n y costos inesperados, poniendo en riesgo su \u00e9xito."}}
Action Result: {'tool_executed': True, 'result': 'Riesgos al lanzar sin pruebas de usuario: 1) Fallos de usabilidad que provocan abandono y mala reputación; 2) Errores críticos no detectados que generan costes de soporte y posibles daños legales; 3) Desalineación con necesidades reales del mercado, resultando en baja adopción y pérdida de inversió

In [ ]:
# patron #2 memrory reflection
# el proceso especialista se copia de vuelta a la memoria del coordinador, pero no se comparte el historial previo del coordinador
resultado = call_agent_with_reflection(action_context, "agente_analista", "Analiza brevemente los riesgos de un lanzamiento apresurado")
print("Resultado:", resultado)
print("Memoria del coordinador ahora tiene:", len(coordinador_memory.items), "items")
for item in coordinador_memory.items:
    print("-", item["type"])

Agent thinking...
Agent Decision: {"tool": "terminate", "args": {"message": "Riesgos de un lanzamiento apresurado: 1) Defectos de calidad que generan fallos y reclamos de usuarios; 2) P\u00e9rdida de reputaci\u00f3n y confianza del cliente; 3) Costos mayores de correcci\u00f3n post\u2011lanzamiento y soporte; 4) Incumplimiento de normativas o requisitos de seguridad; 5) Falta de alineaci\u00f3n con el mercado, provocando bajo rendimiento comercial. En resumen, apresurarse aumenta significativamente los costos y el da\u00f1o reputacional, superando cualquier ventaja de velocidad inmediata."}}
Action Result: {'tool_executed': True, 'result': 'Riesgos de un lanzamiento apresurado: 1) Defectos de calidad que generan fallos y reclamos de usuarios; 2) Pérdida de reputación y confianza del cliente; 3) Costos mayores de corrección post‑lanzamiento y soporte; 4) Incumplimiento de normativas o requisitos de seguridad; 5) Falta de alineación con el mercado, provocando bajo rendimiento comercial. 

In [ ]:
# patron #3 Memory handdoff
# el especialista recibe la memoria completa del coordinador
resultado = hand_off_to_agent(action_context, "agente_analista", "Con base en lo que ya se ha hablado, dame tu conclusión final")
print("Resultado:", resultado)

Agent thinking...
Agent Decision: {"tool": "terminate", "args": {"message": "Conclusi\u00f3n final: Un lanzamiento apresurado conlleva altos riesgos de calidad, costos de correcci\u00f3n y da\u00f1o reputacional que superan los beneficios de velocidad. Es preferible priorizar pruebas, cumplimiento y alineaci\u00f3n con el mercado antes de lanzar, garantizando un producto s\u00f3lido y sostenible."}}
Action Result: {'tool_executed': True, 'result': 'Conclusión final: Un lanzamiento apresurado conlleva altos riesgos de calidad, costos de corrección y daño reputacional que superan los beneficios de velocidad. Es preferible priorizar pruebas, cumplimiento y alineación con el mercado antes de lanzar, garantizando un producto sólido y sostenible.\nTerminando...', 'timestamp': '2026-09-01T01:59:15-0600'}
Resultado: {'result': '{"tool_executed": true, "result": "Conclusi\\u00f3n final: Un lanzamiento apresurado conlleva altos riesgos de calidad, costos de correcci\\u00f3n y da\\u00f1o reputaci

In [ ]:
# patron #4 selective memory sharing
# se arma una memoria de varios temas mezclados, y el LLM elige lo mas relevante para la terea

memoria_mixta = Memory()
for tipo, contenido in [
    ("user", "Necesitamos construir un nuevo dashboard de reportes"),
    ("assistant", "Estimado inicial de costo: $50,000"),
    ("user", "Eso parece mucho"),
    ("assistant", "Desglose: $20k desarrollo, $15k diseño, $15k QA"),
    ("system", "La fecha límite del proyecto se movió a Q3"),
    ("user", "¿Podemos reducir el costo?"),
]:
    memoria_mixta.add_memory({"type": tipo, "content": contenido})

action_context_mixto = ActionContext({
    "llm": generate_response,
    "agent_registry": registry,
    "memory": memoria_mixta
})

resultado = call_agent_with_selected_context(action_context_mixto, "agente_analista", "Revisa el presupuesto y sugiere cómo reducir costos")
print("Resultado:", resultado["result"])
print("Memorias compartidas:", resultado["shared_memories"])
print("Razonamiento de selección:", resultado["selection_reasoning"])

Agent thinking...
Agent Decision: {"tool": "terminate", "args": {"message": "Revisa los gastos fijos y elimina suscripciones no esenciales, renegocia contratos de proveedores, reduce viajes presenciales optando por videollamadas, prioriza compras al por mayor y busca alternativas de bajo costo para insumos. Implementa estas medidas para reducir significativamente el presupuesto."}}
Action Result: {'tool_executed': True, 'result': 'Revisa los gastos fijos y elimina suscripciones no esenciales, renegocia contratos de proveedores, reduce viajes presenciales optando por videollamadas, prioriza compras al por mayor y busca alternativas de bajo costo para insumos. Implementa estas medidas para reducir significativamente el presupuesto.\nTerminando...', 'timestamp': '2026-09-01T01:59:16-0600'}
Resultado: {"tool_executed": true, "result": "Revisa los gastos fijos y elimina suscripciones no esenciales, renegocia contratos de proveedores, reduce viajes presenciales optando por videollamadas, pri